# FRB vs RFI Real-Time CNN Classifier

This notebook provides a complete pipeline for:
1. Loading FRB and RFI datasets
2. Preprocessing and preparing the data
3. Training a CNN classifier
4. Evaluating model performance
5. Testing real-time inference

---

## 1. Setup and Imports

In [1]:
import numpy as np
import h5py
import pickle
import glob
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn.functional as F

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4070


## 2. Configuration

In [9]:
# ============================================================================
# DATA PATHS - UPDATE THESE TO YOUR DATA LOCATIONS
# ============================================================================
FRB_DIR = "realtime_data/FRBs/"  # Directory containing H5 files
RFI_FILE = "realtime_data/RFIs/LOFAR_Full_RFI_dataset.pkl"  # RFI pickle file

# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================
CONFIG = {
    # Data
    'normalize': True,
    'balance_classes': True,
    'test_split': 0.15,
    'val_split': 0.15,
    
    # Model
    'dropout': 0.3,
    
    # Training
    'batch_size': 32,
    'epochs': 50,
    'learning_rate': 0.001,
    'patience': 10,  # Early stopping patience
    
    # Output
    'save_dir': '/home/claude/'
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration:
  normalize: True
  balance_classes: True
  test_split: 0.15
  val_split: 0.15
  dropout: 0.3
  batch_size: 32
  epochs: 50
  learning_rate: 0.001
  patience: 10
  save_dir: /home/claude/


## 3. Data Loading

Load FRB data from H5 files and RFI data from pickle file.

In [ ]:
class FRBDataLoader:
    """Load and preprocess FRB and RFI datasets"""
    
    def __init__(self, frb_dir=None, rfi_file=None):
        self.frb_dir = frb_dir
        self.rfi_file = rfi_file
        self.frb_data = []
        self.rfi_data = []
    
    def load_h5_files(self, file_pattern='*.h5'):
        """Load FRB data from multiple H5 files"""
        print(f"Loading FRB data from {self.frb_dir}...")
        
        h5_files = sorted(glob.glob(str(Path(self.frb_dir) / file_pattern)))
        print(f"Found {len(h5_files)} H5 files")
        
        frb_samples = []
        shapes = []
        
        for i, h5_file in enumerate(h5_files):
            try:
                with h5py.File(h5_file, 'r') as f:
                    # Print structure of first file
                    if i == 0:
                        print(f"\nH5 file structure (first file):")
                        self._print_h5_structure(f)
                    
                    # Try common keys
                    if 'data' in f.keys():
                        data = f['data'][:]
                    elif 'dynamic_spectra' in f.keys():
                        data = f['dynamic_spectra'][:]
                    elif 'freq_time' in f.keys():
                        data = f['freq_time'][:]
                    else:
                        key = list(f.keys())[0]
                        data = f[key][:]
                        if i == 0:
                            print(f"Using key '{key}' from H5 file")
                    
                    frb_samples.append(data)
                    shapes.append(data.shape)
                    
                if (i + 1) % 5 == 0:
                    print(f"Loaded {i + 1}/{len(h5_files)} files...")
                    
            except Exception as e:
                print(f"Error loading {h5_file}: {e}")
                continue
        
        # Determine common shape and pad
        if frb_samples:
            print(f"\nSample shapes encountered: min={min(s[0] for s in shapes)}, max={max(s[0] for s in shapes)}")
            self.frb_data = self._pad_samples(frb_samples, shapes)
        else:
            raise ValueError("No FRB samples loaded successfully")
        
        print(f"\nLoaded FRB data shape: {self.frb_data.shape}")
        return self.frb_data
    
    def _pad_samples(self, samples, shapes):
        """Pad samples to a common shape"""
        # Determine target shape (use max dimensions)
        target_shape = tuple(max(s[i] for s in shapes) for i in range(len(shapes[0])))
        print(f"Padding all samples to shape: {target_shape}")
        
        padded = []
        for sample in samples:
            padded_sample = np.zeros(target_shape, dtype=sample.dtype)
            slices = tuple(slice(0, s) for s in sample.shape)
            padded_sample[slices] = sample
            padded.append(padded_sample)
        
        return np.array(padded)
    
    def load_pkl_file(self):
        """Load RFI data from pickle file"""
        print(f"\nLoading RFI data from {self.rfi_file}...")
        
        with open(self.rfi_file, 'rb') as f:
            rfi_data = pickle.load(f)
        
        # Handle different formats
        if isinstance(rfi_data, dict):
            if 'data' in rfi_data:
                samples = rfi_data['data']
            elif 'rfi' in rfi_data:
                samples = rfi_data['rfi']
            elif 'samples' in rfi_data:
                samples = rfi_data['samples']
            else:
                key = list(rfi_data.keys())[0]
                samples = rfi_data[key]
                print(f"Using key '{key}' from pickle file")
        elif isinstance(rfi_data, list):
            samples = rfi_data
        else:
            samples = rfi_data
        
        # Convert to list if needed
        if not isinstance(samples, list):
            samples = list(samples)
        
        # Check if samples have variable shapes and pad if necessary
        shapes = [np.array(s).shape for s in samples]
        if len(set(shapes)) > 1:
            print(f"Variable shapes detected: {set(shapes)}")
            samples = self._pad_samples(samples, shapes)
        else:
            samples = np.array(samples)
        
        self.rfi_data = samples
        print(f"Loaded RFI data shape: {self.rfi_data.shape}")
        return self.rfi_data
    
    def _print_h5_structure(self, h5_file, indent=0):
        """Recursively print H5 file structure"""
        for key in h5_file.keys():
            item = h5_file[key]
            print('  ' * indent + f"- {key}: ", end='')
            if isinstance(item, h5py.Dataset):
                print(f"Dataset {item.shape} {item.dtype}")
            elif isinstance(item, h5py.Group):
                print(f"Group")
                self._print_h5_structure(item, indent + 1)
    
    def prepare_dataset(self, normalize=True, balance_classes=True):
        """Prepare training dataset from FRB and RFI data"""
        print("\nPreparing dataset...")
        
        # Ensure 3D format
        frb_data = self._ensure_3d(self.frb_data)
        rfi_data = self._ensure_3d(self.rfi_data)
        
        print(f"FRB data shape: {frb_data.shape}")
        print(f"RFI data shape: {rfi_data.shape}")
        
        # Balance classes
        if balance_classes:
            n_frb = len(frb_data)
            n_rfi = len(rfi_data)
            min_samples = min(n_frb, n_rfi)
            
            if n_frb > min_samples:
                indices = np.random.choice(n_frb, min_samples, replace=False)
                frb_data = frb_data[indices]
            if n_rfi > min_samples:
                indices = np.random.choice(n_rfi, min_samples, replace=False)
                rfi_data = rfi_data[indices]
            
            print(f"Balanced to {min_samples} samples per class")
        
        # Create labels
        frb_labels = np.ones(len(frb_data))
        rfi_labels = np.zeros(len(rfi_data))
        
        # Combine
        X = np.vstack([frb_data, rfi_data])
        y = np.concatenate([frb_labels, rfi_labels])
        
        # Shuffle
        indices = np.random.permutation(len(X))
        X = X[indices]
        y = y[indices]
        
        # Normalize
        if normalize:
            print("Normalizing data...")
            X = self._normalize_data(X)
        
        print(f"\nFinal dataset shape: {X.shape}")
        print(f"Class distribution: FRB={np.sum(y==1)}, RFI={np.sum(y==0)}")
        
        return X, y
    
    def _ensure_3d(self, data):
        """Ensure data is 3D"""
        if data.ndim == 2:
            data = np.expand_dims(data, axis=0)
        elif data.ndim == 4:
            n, f, t, p = data.shape
            data = data.reshape(n, f, t * p)
        return data
    
    def _normalize_data(self, X):
        """Normalize data per sample"""
        X_norm = np.zeros_like(X, dtype=np.float32)
        for i in range(len(X)):
            mean = np.mean(X[i])
            std = np.std(X[i])
            if std > 0:
                X_norm[i] = (X[i] - mean) / std
            else:
                X_norm[i] = X[i] - mean
        return X_norm

In [14]:
# Load the data
loader = FRBDataLoader(frb_dir=FRB_DIR, rfi_file=RFI_FILE)

try:
    frb_data = loader.load_h5_files()
    rfi_data = loader.load_pkl_file()
    X, y = loader.prepare_dataset(
        normalize=CONFIG['normalize'],
        balance_classes=CONFIG['balance_classes']
    )
except FileNotFoundError as e:
    print(f"\n❌ Error: {e}")
    print("\nPlease update FRB_DIR and RFI_FILE in the configuration cell above.")
    raise

Loading FRB data from realtime_data/FRBs/...
Found 31 H5 files

H5 file structure (first file):
- first_packet_recv_time: Dataset (903,) float64
- history: Group
- index_map: Group
  - amplitude_norm: Dataset (903, 2) float64
  - beam: Dataset (2,) int64
  - downsampled_time: Dataset (75913,) [('offset_fpga', '<f8'), ('offset_s', '<f8')]
  - freq: Dataset (903,) [('centre', '<f8'), ('id', '<u4')]
  - good_channels: Dataset (760,) uint16
  - input: Dataset (2048,) [('chan_id', '<u2'), ('correlator_input', 'S32')]
  - time: Dataset (75913,) [('offset_fpga', '<u8'), ('offset_s', '<f8')]
- reverse_map: Group
- tiedbeam_baseband: Dataset (903, 2, 75913) complex64
- tiedbeam_locations: Dataset (2,) [('ra', '<f8'), ('dec', '<f8'), ('x_400MHz', '<f8'), ('y_400MHz', '<f8'), ('pol', 'S1')]
- tiedbeam_power: Dataset (903, 2, 75913) float32
- time0: Dataset (903,) [('fpga_count', '<u8'), ('ctime', '<f8'), ('ctime_offset', '<f8')]
Using key 'first_packet_recv_time' from H5 file
Loaded 5/31 files...

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.

## 4. Data Visualization

Visualize sample FRB and RFI dynamic spectra.

In [ ]:
# Plot examples
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Find FRB and RFI examples
frb_indices = np.where(y == 1)[0]
rfi_indices = np.where(y == 0)[0]

# Plot FRBs
for i in range(3):
    idx = frb_indices[i]
    im = axes[0, i].imshow(X[idx], aspect='auto', cmap='viridis', interpolation='nearest')
    axes[0, i].set_title(f'FRB Example {i+1}')
    axes[0, i].set_xlabel('Time bins')
    axes[0, i].set_ylabel('Frequency channels')
    plt.colorbar(im, ax=axes[0, i])

# Plot RFI
for i in range(3):
    idx = rfi_indices[i]
    im = axes[1, i].imshow(X[idx], aspect='auto', cmap='viridis', interpolation='nearest')
    axes[1, i].set_title(f'RFI Example {i+1}')
    axes[1, i].set_xlabel('Time bins')
    axes[1, i].set_ylabel('Frequency channels')
    plt.colorbar(im, ax=axes[1, i])

plt.tight_layout()
plt.savefig(f"{CONFIG['save_dir']}/data_examples.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Data examples saved to {CONFIG['save_dir']}/data_examples.png")

In [ ]:
# Data statistics
print("Dataset Statistics:")
print(f"Total samples: {len(X)}")
print(f"Input shape: {X.shape[1:]}")
print(f"FRB samples: {np.sum(y==1)} ({np.sum(y==1)/len(y)*100:.1f}%)")
print(f"RFI samples: {np.sum(y==0)} ({np.sum(y==0)/len(y)*100:.1f}%)")
print(f"\nData range: [{X.min():.3f}, {X.max():.3f}]")
print(f"Data mean: {X.mean():.3f}")
print(f"Data std: {X.std():.3f}")

## 5. Data Splitting

Split data into training, validation, and test sets.

In [ ]:
# First split: train+val vs test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, 
    test_size=CONFIG['test_split'], 
    random_state=42, 
    stratify=y
)

# Second split: train vs val
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=CONFIG['val_split']/(1-CONFIG['test_split']),
    random_state=42,
    stratify=y_trainval
)

print("Data splits:")
print(f"Train: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  - FRB: {np.sum(y_train==1)} ({np.sum(y_train==1)/len(y_train)*100:.1f}%)")
print(f"  - RFI: {np.sum(y_train==0)} ({np.sum(y_train==0)/len(y_train)*100:.1f}%)")
print(f"\nVal: {len(X_val)} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"  - FRB: {np.sum(y_val==1)} ({np.sum(y_val==1)/len(y_val)*100:.1f}%)")
print(f"  - RFI: {np.sum(y_val==0)} ({np.sum(y_val==0)/len(y_val)*100:.1f}%)")
print(f"\nTest: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")
print(f"  - FRB: {np.sum(y_test==1)} ({np.sum(y_test==1)/len(y_test)*100:.1f}%)")
print(f"  - RFI: {np.sum(y_test==0)} ({np.sum(y_test==0)/len(y_test)*100:.1f}%)")

In [ ]:
# Create PyTorch datasets and dataloaders
train_dataset = TensorDataset(
    torch.FloatTensor(X_train),
    torch.FloatTensor(y_train)
)
val_dataset = TensorDataset(
    torch.FloatTensor(X_val),
    torch.FloatTensor(y_val)
)
test_dataset = TensorDataset(
    torch.FloatTensor(X_test),
    torch.FloatTensor(y_test)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=0
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0
)

print(f"\nCreated data loaders with batch size {CONFIG['batch_size']}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 6. Model Definition

Define the CNN architecture for FRB classification.

In [ ]:
class FRBClassifierCNN(nn.Module):
    """CNN for real-time FRB vs RFI classification"""
    
    def __init__(self, input_shape, dropout=0.3):
        super(FRBClassifierCNN, self).__init__()
        
        freq_channels, time_bins = input_shape
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        # Calculate flattened size
        conv_freq = freq_channels // 8
        conv_time = time_bins // 8
        flat_features = 128 * conv_freq * conv_time
        
        # Fully connected layers
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(flat_features, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 1)
    
    def forward(self, x):
        # Add channel dimension if needed
        if x.ndim == 3:
            x = x.unsqueeze(1)
        
        # Conv blocks
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        
        return x

In [ ]:
# Create model
input_shape = X_train.shape[1:]
model = FRBClassifierCNN(input_shape=input_shape, dropout=CONFIG['dropout']).to(device)

# Model summary
print("Model Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. Training

Train the model with early stopping.

In [ ]:
# Training setup
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'train_acc': [],
    'val_acc': [],
    'learning_rate': []
}

best_val_loss = float('inf')
patience_counter = 0

print("Starting training...")
print("=" * 80)

In [ ]:
# Training loop
for epoch in range(CONFIG['epochs']):
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data).squeeze()
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pred = (torch.sigmoid(output) > 0.5).float()
        train_correct += (pred == target).sum().item()
        train_total += target.size(0)
    
    train_loss /= len(train_loader)
    train_acc = train_correct / train_total
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data).squeeze()
            loss = criterion(output, target)
            
            val_loss += loss.item()
            pred = (torch.sigmoid(output) > 0.5).float()
            val_correct += (pred == target).sum().item()
            val_total += target.size(0)
    
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    
    # Update learning rate
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['learning_rate'].append(current_lr)
    
    # Print progress
    print(f"Epoch [{epoch+1:3d}/{CONFIG['epochs']}] "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
          f"LR: {current_lr:.6f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), f"{CONFIG['save_dir']}/best_model.pth")
        print(f"  → Best model saved (val_loss: {val_loss:.4f})")
    else:
        patience_counter += 1
    
    if patience_counter >= CONFIG['patience']:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

print("\n" + "=" * 80)
print("Training complete!")
print(f"Best validation loss: {best_val_loss:.4f}")

In [ ]:
# Load best model
model.load_state_dict(torch.load(f"{CONFIG['save_dir']}/best_model.pth"))
print("Loaded best model for evaluation")

## 8. Training Visualization

Plot training curves.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Acc', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Learning rate plot
axes[2].plot(history['learning_rate'], linewidth=2, color='green')
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Learning Rate', fontsize=12)
axes[2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['save_dir']}/training_history.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Training curves saved to {CONFIG['save_dir']}/training_history.png")

## 9. Model Evaluation

Comprehensive evaluation on test set.

In [ ]:
# Evaluate on test set
model.eval()

all_preds = []
all_labels = []
all_probs = []

print("Evaluating on test set...")

with torch.no_grad():
    for data, target in test_loader:
        data = data.to(device)
        output = model(data).squeeze()
        probs = torch.sigmoid(output).cpu().numpy()
        preds = (probs > 0.5).astype(int)
        
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(target.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

print("Evaluation complete!\n")

In [ ]:
# Classification metrics
print("=" * 80)
print("TEST SET EVALUATION RESULTS")
print("=" * 80)
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['RFI', 'FRB'], digits=4))

# Overall accuracy
accuracy = np.mean(all_preds == all_labels)
print(f"\nOverall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# ROC AUC
try:
    auc = roc_auc_score(all_labels, all_probs)
    print(f"ROC AUC Score: {auc:.4f}")
except:
    auc = None
    print("ROC AUC: Could not compute")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix - counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['RFI', 'FRB'],
            yticklabels=['RFI', 'FRB'],
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('True', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')

# Confusion matrix - normalized
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=['RFI', 'FRB'],
            yticklabels=['RFI', 'FRB'],
            ax=axes[1], cbar_kws={'label': 'Proportion'})
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('True', fontsize=12)
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{CONFIG['save_dir']}/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nConfusion matrix saved to {CONFIG['save_dir']}/confusion_matrix.png")

In [ ]:
# ROC Curve
if auc is not None:
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {auc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curve', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/roc_curve.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"ROC curve saved to {CONFIG['save_dir']}/roc_curve.png")

In [ ]:
# Prediction confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of all predictions
axes[0].hist(all_probs[all_labels==0], bins=50, alpha=0.6, label='RFI (True)', color='blue')
axes[0].hist(all_probs[all_labels==1], bins=50, alpha=0.6, label='FRB (True)', color='red')
axes[0].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
axes[0].set_xlabel('Predicted Probability (FRB)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Prediction Confidence Distribution', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Box plot
data_to_plot = [all_probs[all_labels==0], all_probs[all_labels==1]]
bp = axes[1].boxplot(data_to_plot, labels=['RFI', 'FRB'], patch_artist=True)
bp['boxes'][0].set_facecolor('blue')
bp['boxes'][1].set_facecolor('red')
axes[1].axhline(y=0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
axes[1].set_ylabel('Predicted Probability (FRB)', fontsize=12)
axes[1].set_title('Prediction Confidence by True Class', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{CONFIG['save_dir']}/prediction_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Prediction distribution saved to {CONFIG['save_dir']}/prediction_distribution.png")

## 10. Error Analysis

Analyze misclassified examples.

In [ ]:
# Find misclassified examples
misclassified_idx = np.where(all_preds != all_labels)[0]
print(f"Total misclassifications: {len(misclassified_idx)} / {len(all_labels)} ({len(misclassified_idx)/len(all_labels)*100:.2f}%)")

# False positives (predicted FRB, actually RFI)
fp_idx = np.where((all_preds == 1) & (all_labels == 0))[0]
print(f"False Positives (RFI classified as FRB): {len(fp_idx)}")

# False negatives (predicted RFI, actually FRB)
fn_idx = np.where((all_preds == 0) & (all_labels == 1))[0]
print(f"False Negatives (FRB classified as RFI): {len(fn_idx)}")

In [ ]:
# Visualize some misclassified examples
if len(misclassified_idx) > 0:
    n_show = min(6, len(misclassified_idx))
    sample_idx = np.random.choice(misclassified_idx, n_show, replace=False)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    for i, idx in enumerate(sample_idx):
        true_label = 'FRB' if all_labels[idx] == 1 else 'RFI'
        pred_label = 'FRB' if all_preds[idx] == 1 else 'RFI'
        prob = all_probs[idx]
        
        im = axes[i].imshow(X_test[idx], aspect='auto', cmap='viridis', interpolation='nearest')
        axes[i].set_title(f'True: {true_label} | Pred: {pred_label} (p={prob:.3f})', fontsize=10)
        axes[i].set_xlabel('Time bins')
        axes[i].set_ylabel('Freq channels')
        plt.colorbar(im, ax=axes[i])
    
    # Hide extra subplots if any
    for i in range(n_show, 6):
        axes[i].axis('off')
    
    plt.suptitle('Misclassified Examples', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/misclassified_examples.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Misclassified examples saved to {CONFIG['save_dir']}/misclassified_examples.png")
else:
    print("No misclassifications to visualize!")

## 11. Inference Latency Benchmark

Measure real-time inference speed.

In [ ]:
# Benchmark inference latency
model.eval()

# Warm up
with torch.no_grad():
    dummy = torch.randn(1, *input_shape).to(device)
    for _ in range(10):
        _ = model(dummy)

# Benchmark single sample inference
n_runs = 1000
latencies_single = []

print(f"Benchmarking single-sample inference ({n_runs} runs)...")

for i in range(n_runs):
    sample = torch.randn(1, *input_shape).to(device)
    
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(sample)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    end = time.perf_counter()
    
    latencies_single.append((end - start) * 1000)  # Convert to ms

latencies_single = np.array(latencies_single)

print("\nSingle-sample inference latency:")
print(f"  Mean: {latencies_single.mean():.3f} ms")
print(f"  Median: {np.median(latencies_single):.3f} ms")
print(f"  Std: {latencies_single.std():.3f} ms")
print(f"  Min: {latencies_single.min():.3f} ms")
print(f"  Max: {latencies_single.max():.3f} ms")
print(f"  95th percentile: {np.percentile(latencies_single, 95):.3f} ms")
print(f"  99th percentile: {np.percentile(latencies_single, 99):.3f} ms")

In [ ]:
# Benchmark batch inference
batch_sizes = [1, 8, 16, 32, 64]
batch_results = {}

print("\nBenchmarking batch inference...")

for bs in batch_sizes:
    latencies = []
    
    for _ in range(100):
        batch = torch.randn(bs, *input_shape).to(device)
        
        start = time.perf_counter()
        with torch.no_grad():
            _ = model(batch)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        end = time.perf_counter()
        
        latencies.append((end - start) * 1000)
    
    latencies = np.array(latencies)
    batch_results[bs] = {
        'total_ms': latencies.mean(),
        'per_sample_ms': latencies.mean() / bs
    }
    
    print(f"Batch size {bs:2d}: {latencies.mean():.3f} ms total, "
          f"{latencies.mean()/bs:.3f} ms per sample")

In [ ]:
# Plot latency results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Latency distribution
axes[0].hist(latencies_single, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(latencies_single.mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {latencies_single.mean():.3f} ms')
axes[0].axvline(np.median(latencies_single), color='green', linestyle='--',
                linewidth=2, label=f'Median: {np.median(latencies_single):.3f} ms')
axes[0].set_xlabel('Latency (ms)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Single-Sample Inference Latency Distribution', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Batch size vs per-sample latency
batch_sizes_plot = list(batch_results.keys())
per_sample_latencies = [batch_results[bs]['per_sample_ms'] for bs in batch_sizes_plot]

axes[1].plot(batch_sizes_plot, per_sample_latencies, marker='o', markersize=8, linewidth=2)
axes[1].set_xlabel('Batch Size', fontsize=12)
axes[1].set_ylabel('Per-Sample Latency (ms)', fontsize=12)
axes[1].set_title('Batch Size vs Per-Sample Latency', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log', base=2)

plt.tight_layout()
plt.savefig(f"{CONFIG['save_dir']}/inference_latency.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nLatency analysis saved to {CONFIG['save_dir']}/inference_latency.png")

## 12. Save Final Model and Results

Save the trained model and evaluation results.

In [ ]:
# Save final model
torch.save(model.state_dict(), f"{CONFIG['save_dir']}/frb_classifier_final.pth")
print(f"Final model saved to {CONFIG['save_dir']}/frb_classifier_final.pth")

# Save model metadata
model_info = {
    'input_shape': input_shape,
    'config': CONFIG,
    'total_params': sum(p.numel() for p in model.parameters()),
    'test_accuracy': accuracy,
    'test_auc': auc,
    'mean_inference_latency_ms': latencies_single.mean(),
    'median_inference_latency_ms': np.median(latencies_single),
    'training_epochs': len(history['train_loss']),
    'best_val_loss': best_val_loss
}

with open(f"{CONFIG['save_dir']}/model_info.pkl", 'wb') as f:
    pickle.dump(model_info, f)

print(f"Model info saved to {CONFIG['save_dir']}/model_info.pkl")

# Save evaluation results
results = {
    'predictions': all_preds,
    'labels': all_labels,
    'probabilities': all_probs,
    'confusion_matrix': cm,
    'accuracy': accuracy,
    'auc': auc
}

with open(f"{CONFIG['save_dir']}/evaluation_results.pkl", 'wb') as f:
    pickle.dump(results, f)

print(f"Evaluation results saved to {CONFIG['save_dir']}/evaluation_results.pkl")

## 13. Summary

Final summary of model performance.

In [ ]:
print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print("\nModel:")
print(f"  Architecture: CNN with 3 conv blocks + 3 FC layers")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Input shape: {input_shape}")

print("\nTraining:")
print(f"  Epochs completed: {len(history['train_loss'])}")
print(f"  Best validation loss: {best_val_loss:.4f}")
print(f"  Final training accuracy: {history['train_acc'][-1]:.4f}")
print(f"  Final validation accuracy: {history['val_acc'][-1]:.4f}")

print("\nTest Performance:")
print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
if auc:
    print(f"  ROC AUC: {auc:.4f}")
print(f"  False Positives: {len(fp_idx)} ({len(fp_idx)/len(all_labels)*100:.2f}%)")
print(f"  False Negatives: {len(fn_idx)} ({len(fn_idx)/len(all_labels)*100:.2f}%)")

print("\nInference Speed:")
print(f"  Mean latency: {latencies_single.mean():.3f} ms")
print(f"  Median latency: {np.median(latencies_single):.3f} ms")
print(f"  95th percentile: {np.percentile(latencies_single, 95):.3f} ms")
print(f"  Throughput (single): ~{1000/latencies_single.mean():.1f} samples/sec")
if 32 in batch_results:
    print(f"  Throughput (batch=32): ~{32000/batch_results[32]['total_ms']:.1f} samples/sec")

print("\nSaved Files:")
print(f"  Model: {CONFIG['save_dir']}/frb_classifier_final.pth")
print(f"  Best model: {CONFIG['save_dir']}/best_model.pth")
print(f"  Model info: {CONFIG['save_dir']}/model_info.pkl")
print(f"  Evaluation results: {CONFIG['save_dir']}/evaluation_results.pkl")
print(f"  Training history plot: {CONFIG['save_dir']}/training_history.png")
print(f"  Confusion matrix: {CONFIG['save_dir']}/confusion_matrix.png")
print(f"  ROC curve: {CONFIG['save_dir']}/roc_curve.png")
print(f"  Data examples: {CONFIG['save_dir']}/data_examples.png")

print("\n" + "=" * 80)
print("✓ Training and evaluation complete!")
print("=" * 80)